In [ ]:
import numpy as np
import sdeint
import tqdm
import os
import matplotlib.pyplot as plt

In [ ]:
# Parameters
N = 0.44 * 1e12  # Number of atoms
q = 0.198        # Parameter
T2 = 0.87 * 1e-3  # Relaxation time in s
g_D = 0.00177    # [pA]
Sph = 96         # Photon noise PSD [pA^2/Hz]
w0 = 2 * np.pi * 1e4  # Mean value of the Larmor in OU process
w01 = w0 * T2  # in non dimensional units
h = 50 * 1e-9    # Time step is the solver time step not the probing time!whath out
tf = 2.        # Final time in [ms]
nsteps = int(tf * 1e-3 / h)  # Number of steps in simulation
h1 = h / T2      # Time step in non-dimensional units
dc = 0.0
# dc = 0.01         # Diffusion constant of the OU process 0.01 for fluctuating
tau = 0.001
# tau=0.1
measure_every_nth = 100
meas_probing_rate = h*measure_every_nth  # in s
#TRANSFORM DATA - compute coefficients
Nq = N * q
xc = np.sqrt(2 / Nq)
yc = np.sqrt(2 / (g_D**2 * Nq))
sig_v = np.sqrt(2 * Sph / (g_D**2 * Nq * meas_probing_rate))
meas_probing_rate_unitless = meas_probing_rate/T2

# Time constant of the OU process in non-dimensional units
G = np.diag([np.sqrt(2), np.sqrt(2), np.sqrt(dc)])  # Matrix G
D = np.array([[2, 0, 0], [0, 2, 0], [0, 0, dc]])

xc2 = xc*xc
sgv2 = sig_v**2  # measurement noise variance

# Time vector in [ms] simulation
t = np.arange(0, tf/(T2*1e3), h1)
t_meas = np.arange(0, tf/(T2*1e3), meas_probing_rate_unitless)

# Initial condition (simulation)
x = np.array([0, np.sqrt(2 / (q * N)) * 0.5 * N, w0 * T2])


In [ ]:
# SIMULATION
def f_x(x, t):
    f = np.zeros(3)
    f[0] += -x[0] + x[2] * x[1]
    f[1] += -x[2]*x[0] - x[1]
    # f[2] += (w01 - x[2]) / tau  # OU process
    f[2] += 0
    return f

def f_x_OU(x, t):
    f = np.zeros(3)
    f[0] += -x[0] + x[2] * x[1]
    f[1] += -x[2]*x[0] - x[1]
    f[2] += (w01 - x[2]) / tau  # OU process
    return f

def get_intrinsic_noise(dt):
    """Generates dW Wiener increment of the intrinsic noise."""
    return np.array([np.sqrt(dt),
                     np.sqrt(dt),
                     np.sqrt(dt)]) * np.random.randn(3)

def get_G(x, t):
    return np.diag([np.sqrt(2), np.sqrt(2), np.sqrt(dc)])

def step(x0, t, dt, num_steps=20, type=None):
    t_span = np.linspace(t, t + dt, num_steps)
    dW = np.array([get_intrinsic_noise(dt) for _ in t_span[1:]])
    if type==None:
        x = sdeint.itoSRI2(f=f_x,
                           G=get_G,
                           y0=x0,
                           tspan=t_span,
                           dW=dW)
    elif type=="OU":
        x = sdeint.itoSRI2(f=f_x_OU,
                           G=get_G,
                           y0=x0,
                           tspan=t_span,
                           dW=dW)
    elif type=="jump":
        if t <= 2.29879/8:
            x0[2] = 2*np.pi*9800*T2
           
        elif (t>=  2.29879/8 and t<=2.29879/4):
            x0[2] = 2*np.pi*9100.4*T2
        
        else:
            x0[2] = 2*np.pi*10000.0*T2
            
        x = sdeint.itoSRI2(f=f_x,
                           G=get_G,
                           y0=x0,
                           tspan=t_span,
                           dW=dW)
    elif type=="sine":
        x0[2] = 0.1*w01*np.sin(0.05*w01*t)+ 2*np.pi*10800.0*T2
        x = sdeint.itoSRI2(f=f_x,
                           G=get_G,
                           y0=x0,
                           tspan=t_span,
                           dW=dW)
    return x[-1]

def simulate(t_max, dt, x0, xs, num_steps=20, type=None):
    time_arr = np.arange(0, t_max, dt)
    for index, time in enumerate(tqdm.tqdm(time_arr, desc='pid:%r' % os.getpid())):
        # SIMULATION AND MEASUREMENT==============================
        x = step(x0, time, dt, num_steps, type=type)
        xs[index] = x
        x0 = x         
    return xs


In [ ]:
time_arr_sim_unitless = np.arange(0, tf/(T2*1e3), h1)
xs = np.array([np.zeros_like(x) for _ in time_arr_sim_unitless])
simulate(tf/(T2*1e3), h1, x, xs)

In [ ]:
#Plot simulation
fig, axs = plt.subplots(3, 1, layout='constrained')
axs[0].plot(t, xs[:,0], label="J_y")
axs[0].set_xlabel('Time (s)')
axs[0].set_ylabel('J_y')
axs[0].grid(True)

axs[1].plot(t, xs[:,1], label="J_z")
axs[1].set_xlabel('Time (s)')
axs[1].set_ylabel('J_z')
axs[1].grid(True)

axs[2].plot(t, xs[:,2], label="J_y")
axs[2].set_xlabel('Time (s)')
axs[2].set_ylabel('J_y')
axs[2].grid(True)

# Generate noisy measurement outcomes

In [ ]:
#Generate measurement outcomes
yh = []
for idx, x1_val in enumerate(xs[:, 1]):
    if idx % measure_every_nth == 0:  # Collect every nth sample
        yh.append(x1_val + sig_v * np.random.randn())  # Add noise to the measurement

In [ ]:
plt.plot(t_meas, yh)
plt.title("Measurement outcomes")
plt.show()

# EKF

In [ ]:
def dP_dt(t, P, x, Q, F, dim_x):
    P_matrix = np.reshape(P, (dim_x, dim_x))
    return np.reshape(F(x,t)@ P_matrix + P_matrix@F(x,t).T + Q,dim_x**2)
    # return np.reshape(np.dot(F(x, t), np.reshape(P, (dim_x, dim_x))) +
    #                   np.dot(np.reshape(P, (dim_x, dim_x)), np.transpose(F(x, t)) + Q),
    #                   dim_x ** 2)


def F(x, t):
    # Jacobian of f
    return np.array([
        [-1, x[2], x[1]],
        [-x[2], -1, -x[0]],
        [0, 0, 0]
        # [0, 0, -1 / tau]
    ])

def F_OU(x, t):
    # Jacobian of f
    return np.array([
        [-1, x[2], x[1]],
        [-x[2], -1, -x[0]],
        [0, 0, -1 / tau]
    ])

def f_x_ekf(t, x):
    f = np.zeros(3)
    f[0] += -x[0] + x[2] * x[1]
    f[1] += -x[2] * x[0] - x[1]
    # f[2] += (w01 - x[2]) / tau  # OU process
    f[2] += 0
    return f

def f_x_OU_ekf(t, x):
    f = np.zeros(3)
    f[0] += -x[0] + x[2] * x[1]
    f[1] += -x[2] * x[0] - x[1]
    f[2] += (w01 - x[2]) / tau  # OU process
    return f

def ekf_predict(t, x, P, Q, delta_t, dim_x, type=None):
    if type == None:
        P_sol = solve_ivp(dP_dt,
                          [t, t + delta_t],
                          np.reshape(P, dim_x ** 2),
                          method='RK45',
                          dense_output=True,
                          max_step=delta_t/(measure_every_nth*20),
                          args=(x,
                                Q,
                                F,
                                dim_x))
        x_sol = solve_ivp(f_x_ekf,
                          [t, t + delta_t],
                          x,
                          method='RK45',
                          max_step=delta_t/(measure_every_nth*20),
                          dense_output=True)
    if type == "OU":
        P_sol = solve_ivp(dP_dt,
                          [t, t + delta_t],
                          np.reshape(P, dim_x ** 2),
                          method='RK45',
                          dense_output=True,
                          max_step=delta_t/(measure_every_nth*10),
                          args=(x,
                                Q,
                                F_OU,
                                dim_x))
        x_sol = solve_ivp(f_x_OU_ekf,
                      [t, t + delta_t],
                      x,
                      method='RK45',
                      max_step=delta_t/(measure_every_nth*10),
                      dense_output=True)
    
    P_temp = P_sol.sol(t + delta_t)
    x = x_sol.sol(t + delta_t)
    P = np.reshape(P_temp, (dim_x, dim_x))
    # P = 0.5 * (P + P.T)
    return x, P


def ekf_update_new(y, x, P, R_delta):
    """
    Perform one EKF update step given the measurement y.

    Args:
        y: Scalar measurement value.
        x: Current state estimate (dim_x-dimensional vector).
        P: Current state covariance matrix (dim_x x dim_x).
        R_delta: Measurement noise covariance (scalar).

    Returns:
        Updated state estimate (x) and covariance (P).
    """
    H = np.array([[0., 1., 0.]])  # Shape: (1, dim_x)

    # Compute innovation (difference between actual and predicted measurement)
    innovation = np.array([y]) - np.dot(H, x)  # Shape: (1,)

    # Innovation covariance
    S = np.dot(H, np.dot(P, H.T)) + R_delta  # Shape: (1, 1)

    # Kalman gain
    K = np.dot(P, np.dot(H.T, np.linalg.inv(S)))  # Shape: (dim_x, 1)

    # State update
    x_new = x + np.dot(K, innovation)

    P_new = (np.eye(3)-K@H)@P
    P_new = 0.5*(P_new+P_new.T)
    return x_new, P_new

In [ ]:
# Kalman initialization
dw0 = 0.01
dJ = 0.01
x_current = np.array([0, 0.5 * xc * N * (1 + dJ), w0 * (1 + dw0)*T2])
P_current = np.diag([
    (0.5 * xc * N * dJ/3)**2,
    (0.5 * xc * N * dJ/3)**2,
    (w0 * T2 * dw0/3)**2
])

# Time vector in [ms] simulation
t = np.arange(0, len(xs[:, 2])) * h1
t_meas = np.arange(0, len(yh)) * meas_probing_rate_unitless

# Allocate memory
P_est = np.zeros((len(yh), 3))
x_est = np.zeros((len(yh), 3))
print(x_current, xs[0, :], "initial estimate")

for index, val in enumerate(tqdm.tqdm(t_meas, desc='pid:%r' % os.getpid())):
    x_current, P_current = ekf_update_new(yh[index], x_current, P_current, R_delta=sgv2)
    x_est[index, :] = x_current.T  # Store the state estimate (transpose if m is a row vector)
    P_est[index, :] = [P_current[0, 0], P_current[1, 1], P_current[2, 2]]
    x_current, P_current = ekf_predict(val, x_current, P_current, D, delta_t=meas_probing_rate_unitless, dim_x=3)

In [ ]:
# Larmor frequency
f_L = xs[:, 2] / (2 * np.pi * T2 * 1e3)  # [kHz]
f_ud = (w0 * T2 + 3 * np.sqrt(dc) * np.array([-1, 1])) / (2 * np.pi * T2 * 1e3)  # 3σ bounds
#
# Plot Larmor frequency
plt.plot(t_meas, x_est[:, 1], label='EKF')
plt.plot(t, xs[:, 1], label='Jz')
# plt.axhline(f_ud[0], linestyle='--', color='red', label='Lower Bound')
# plt.axhline(f_ud[1], linestyle='--', color='green', label='Upper Bound')
plt.xlabel('t [ms]')
plt.ylabel('J_z')
plt.title('Jz, OU process')
plt.legend()
plt.show()
#
plt.plot(t_meas, x_est[:, 0])
plt.plot(t, xs[:, 0])
plt.title('J_y')
plt.legend()
plt.show()

plt.plot(t_meas, x_est[:, 2])
plt.plot(t, xs[:, 2])
plt.show()

# Simulation time varying field

In [ ]:
# dc = 0.001 # we need to change dc only because it was 0 for const field 
dc = 1e-2
tf= 1.
tau = 1e3
time_arr_sim_unitless_OU = np.arange(0, tf/(T2*1e3), h1)
xs_OU = np.array([np.zeros_like(x) for _ in time_arr_sim_unitless_OU])
simulate(tf/(T2*1e3), h1, x, xs_OU, type="OU")

get_G(x, 0)

In [ ]:
#Plot simulation
fig, axs = plt.subplots(3, 1, layout='constrained')
axs[0].plot(t, xs_OU[:, 0], label="J_y")
axs[0].set_xlabel('Time (s)')
axs[0].set_ylabel('J_y')
axs[0].grid(True)

axs[1].plot(t, xs_OU[:, 1], label="J_z")
axs[1].set_xlabel('Time (s)')
axs[1].set_ylabel('J_z')
axs[1].grid(True)

axs[2].plot(t, xs_OU[:, 2], label="J_y")
axs[2].set_xlabel('Time (s)')
axs[2].set_ylabel('J_y')
axs[2].grid(True)
t = np.arange(0, 1 / (T2 * 1e3), h1)

plt.plot(t, xs_OU[:, 2], label="J_y")

In [ ]:
import pandas as pd
# Save full simulation data
OU_sim_data = pd.DataFrame({
    "time_sim": t*T2,
    "J_y_sim": xs_OU[:, 0]/xc,
    "J_z_sim": xs_OU[:, 1]/xc,
    "omega_sim": xs_OU[:, 2]/T2
})
OU_sim_data.to_csv("C:/Users/Klaudia/Documents/CD_EKF/OU_simulation_dc_%f_tau_%dtau" %(dc, tau))

In [ ]:
#Generate measurement outcomes
measure_every_nth = 100
meas_probing_rate= h*measure_every_nth
meas_probing_rate_unitless = meas_probing_rate/T2
yh = []
x_sim_OU_meas_freq = []

for idx, x1_val in enumerate(xs_OU[:, 1]):
    if idx % measure_every_nth == 0:  # Collect every nth sample
        yh.append(x1_val + sig_v * np.random.randn())
        x_sim_OU_meas_freq.append(xs_OU[idx,:])

x_sim_OU_meas_freq = np.array(x_sim_OU_meas_freq)

In [ ]:
# Kalman initialization
dw0 = 0.01
dJ = 0.01
dc = dc
D_OU = get_G(x,0)@get_G(x, 0).T
x_current_OU = np.array([0, 0.5 * xc * N * (1 + dJ), w0 * (1 + dw0)*T2])
P_current_OU = np.diag([
    (0.5 * xc * N * dJ/3)**2,
    (0.5 * xc * N * dJ/3)**2,
    (w0 * T2 * dw0/3)**2
])

# Time vector in [ms] simulation
t_OU = np.arange(0, len(xs_OU[:, 2])) * h1
t_meas_OU = np.arange(0, len(yh)) * meas_probing_rate_unitless

# Allocate memory
P_est_OU = np.zeros((len(yh), 3))
x_est_OU = np.zeros((len(yh), 3))

print(x_current_OU, xs_OU[0, :], "initial estimate")
print(D_OU)

for index, val in enumerate(tqdm.tqdm(t_meas_OU, desc='pid:%r' % os.getpid())):
    x_current_OU, P_current_OU = ekf_update_new(yh[index], x_current_OU, P_current_OU, R_delta=sgv2)
    x_est_OU[index, :] = x_current_OU.T  
    P_est_OU[index, :] = [P_current_OU[0, 0], P_current_OU[1, 1], P_current_OU[2, 2]]
    # x_current_OU, P_current_OU = ekf_predict(val, x_current_OU, P_current_OU, D_OU, delta_t=meas_probing_rate_unitless, dim_x=3)
    x_current_OU, P_current_OU = ekf_predict(val, x_current_OU, P_current_OU, D_OU, delta_t=meas_probing_rate_unitless, dim_x=3, type="OU")

In [ ]:
#Plot simulation
fig, axs = plt.subplots(3, 1, layout='constrained')
axs[0].plot(t, xs_OU[:,0], label="J_y")
axs[0].plot(t_meas_OU, x_est_OU[:, 0], label="EKF")
axs[0].set_xlabel('Time (s)')
axs[0].set_ylabel('J_y')
axs[0].grid(True)

axs[1].plot(t, xs_OU[:,1], label="J_z")
axs[1].plot(t_meas_OU, x_est_OU[:, 1], label="EKF")
axs[1].set_xlabel('Time (s)')
axs[1].set_ylabel('J_z')
axs[1].grid(True)

axs[2].plot(t, xs_OU[:,2], label="freq")
axs[2].plot(t_meas_OU, x_est_OU[:, 2], label="EKF")
axs[2].set_xlabel('Time (s)')
axs[2].set_ylabel('freq')
axs[2].grid(True)

In [ ]:
OU_EKF_data = pd.DataFrame({
    "time": t_meas_OU*T2,
    "J_y_EKF": x_est_OU[:, 0]/xc,
    "J_z_EKF": x_est_OU[:, 1]/xc,
    "omega_EKF": x_est_OU[:, 2]/T2,
    "P_J_y": P_est_OU[:, 0]/xc2,
    "P_J_z": P_est_OU[:, 1]/xc2,
    "P_omega": P_est_OU[:, 2]/T2**2,
    "J_y": x_sim_OU_meas_freq[:, 0]/xc,
    "J_z": x_sim_OU_meas_freq[:, 1]/xc,
    "omega": x_sim_OU_meas_freq[:, 2]/T2
})
OU_EKF_data.to_csv("CD_EKF/OU_EKF_RELAXATION_dc_%f_tau_%f_delta_%f" %(dc, tau, meas_probing_rate))

# Jump Simulation

In [ ]:
dc = 0.0
time_arr_sim_unitless = np.arange(0, tf/(T2*1e3), h1)
xs = np.array([np.zeros_like(x) for _ in time_arr_sim_unitless])
simulate(tf/(T2*1e3), h1, x, xs, type="jump")

In [ ]:
measure_every_nth = 10
meas_probing_rate= h*measure_every_nth
meas_probing_rate_unitless = meas_probing_rate/T2
yh = []
x_sim_jump_meas_freq = []
for idx, x1_val in enumerate(xs[:, 1]):
    if idx % measure_every_nth == 0:  # Collect every nth sample
        yh.append(x1_val + sig_v * np.random.randn())
        x_sim_jump_meas_freq.append(xs[idx,:])

x_sim_jump_meas_freq = np.array(x_sim_jump_meas_freq)# Add noise to the measurement

In [ ]:
# Kalman initialization
dw0 = 0.01
dJ = 0.01
dc = 0.01

x_current = np.array([0, 0.5 * xc * N * (1 + dJ), w0 * (1 + dw0)*T2])
P_current = np.diag([
    (0.5 * xc * N * dJ/3)**2,
    (0.5 * xc * N * dJ/3)**2,
    (w0 * T2 * dw0/3)**2
])
D = get_G(x_current, t)@get_G(x_current, t).T
# Time vector in [ms] simulation
t = np.arange(0, len(xs[:, 2])) * h1
t_meas = np.arange(0, len(yh)) * meas_probing_rate_unitless

# Allocate memory
P_est = np.zeros((len(yh), 3))
x_est = np.zeros((len(yh), 3))
print(x_current, xs[0, :], "initial estimate")

for index, val in enumerate(tqdm.tqdm(t_meas, desc='pid:%r' % os.getpid())):
    x_current, P_current = ekf_update_new(yh[index], x_current, P_current, R_delta=sgv2)
    x_est[index, :] = x_current.T  # Store the state estimate (transpose if m is a row vector)
    P_est[index, :] = [P_current[0, 0], P_current[1, 1], P_current[2, 2]]
    x_current, P_current = ekf_predict(val, x_current, P_current, D, delta_t=meas_probing_rate_unitless, dim_x=3)

In [ ]:
plt.plot(t_meas, x_est[:, 2])
plt.plot(time_arr_sim_unitless, xs[:, 2])
# plt.ylim(53.55, 53.60)
plt.show()

In [ ]:
jump_EKF_data = pd.DataFrame({
    "time": t_meas*T2,
    "J_y_EKF": x_est[:, 0]/xc,
    "J_z_EKF": x_est[:, 1]/xc,
    "omega_EKF": x_est[:, 2]/T2,
    "P_J_y": P_est[:, 0]/xc2,
    "P_J_z": P_est[:, 1]/xc2,
    "P_omega": P_est[:, 2]/T2**2,
    "J_y": x_sim_jump_meas_freq[:, 0]/xc,
    "J_z": x_sim_jump_meas_freq[:, 1]/xc,
    "omega": x_sim_jump_meas_freq[:, 2]/T2
})
jump_EKF_data.to_csv("jump_EKF_dc_%f_tau_%f_delta_%f" %(dc, tau, meas_probing_rate))

# Sine signal

In [ ]:
dc = 0.0
tf=2.0

In [ ]:
time_arr_sim_unitless = np.arange(0, tf/(T2*1e3), h1)
xs_sine = np.array([np.zeros_like(x) for _ in time_arr_sim_unitless])
simulate(tf/(T2*1e3), h1, x, xs_sine, type="sine")

In [ ]:
measure_every_nth = 20
meas_probing_rate= h*measure_every_nth
meas_probing_rate_unitless = meas_probing_rate/T2
yh_sine = []
x_sim_sin_meas_freq = []
for idx, x1_val in enumerate(xs_sine[:, 1]):
    if idx % measure_every_nth == 0:  # Collect every nth sample
        yh_sine.append(x1_val + sig_v * np.random.randn())
        x_sim_sin_meas_freq.append(xs_sine[idx,:])

x_sim_sin_meas_freq = np.array(x_sim_sin_meas_freq)

In [ ]:
# Kalman initialization
# dw0 = 0.01
dw0 = 1.1
dJ = 0.01
dc = 0.0001

x_current = np.array([0, 0.5 * xc * N * (1 + dJ), w0 * (1 + dw0) * T2])
P_current = np.diag([
    (0.5 * xc * N * dJ / 3) ** 2,
    (0.5 * xc * N * dJ / 3) ** 2,
    (w0 * T2 / 3) ** 2
])
D = get_G(x_current, t) @ get_G(x_current, t).T
# Time vector in [ms] simulation
t = np.arange(0, len(xs_sine[:, 2])) * h1
t_meas = np.arange(0, len(yh_sine)) * meas_probing_rate_unitless

# Allocate memory
P_est = np.zeros((len(yh_sine), 3))
x_est = np.zeros((len(yh_sine), 3))
print(x_current, xs_sine[0, :], "initial estimate")

for index, val in enumerate(tqdm.tqdm(t_meas, desc='pid:%r' % os.getpid())):
    x_current, P_current = ekf_update_new(yh_sine[index], x_current, P_current, R_delta=sgv2)
    x_est[index, :] = x_current.T  # Store the state estimate (transpose if m is a row vector)
    P_est[index, :] = [P_current[0, 0], P_current[1, 1], P_current[2, 2]]
    x_current, P_current = ekf_predict(val, x_current, P_current, D, delta_t=meas_probing_rate_unitless, dim_x=3)
plt.plot(t_meas, x_est[:, 2])
plt.plot(t, xs_sine[:, 2])
# plt.ylim(50,60)
plt.show()

In [ ]:
sin_EKF_data = pd.DataFrame({
    "time": t_meas*T2,
    "J_y_EKF": x_est[:, 0]/xc,
    "J_z_EKF": x_est[:, 1]/xc,
    "omega_EKF": x_est[:, 2]/T2,
    "P_J_y": P_est[:, 0]/xc2,
    "P_J_z": P_est[:, 1]/xc2,
    "P_omega": P_est[:, 2]/T2**2,
    "J_y": x_sim_sin_meas_freq[:, 0]/xc,
    "J_z": x_sim_sin_meas_freq[:, 1]/xc,
    "omega": x_sim_sin_meas_freq[:, 2]/T2
})
sin_EKF_data.to_csv("C:/Users/Klaudia/Documents/CD_EKF/sin_EKF_dc_%f_tau_%f_delta_%f_new1" %(dc, tau, meas_probing_rate))